In [1]:
%pip -q install fasttext-wheel

In [2]:
from __future__ import annotations

import json
import math
import random
import re
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Iterable

import fasttext
import pandas as pd
import requests
from IPython.display import display
from google.colab import userdata

In [3]:
PARQUET_PATH = Path("/content/sessions_lang_transcript.parquet")
FASTTEXT_MODEL_PATH = Path("/content/lid.176.ftz")
FASTTEXT_MODEL_URL = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz"

OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_MODEL = "openai/gpt-5.4"

TARGET_SESSION_ID = 140595413
SEGMENT_LIMIT = None

FASTTEXT_TOP_K = 3
FASTTEXT_WINDOW_SIZE = 8
FASTTEXT_WINDOW_STRIDE = 4
FASTTEXT_SEGMENT_CONFIDENCE = 0.80
FASTTEXT_WINDOW_CONFIDENCE = 0.80

MIN_MISMATCH_WORDS = 2
MIN_MISMATCH_PERCENTAGE = 10.0

REQUEST_TIMEOUT_SECONDS = 120
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 2

PARQUET_COLUMNS = [
    "gamesession_id",
    "game_name",
    "model_type",
    "lang_detected",
    "lang_probability",
    "transcript_segments",
]

LANGUAGE_ALIASES = {
    "english": "en",
    "russian": "ru",
    "indonesian": "id",
    "german": "de",
    "french": "fr",
    "spanish": "es",
    "portuguese": "pt",
    "ukrainian": "uk",
    "polish": "pl",
    "japanese": "ja",
    "korean": "ko",
    "chinese": "zh",
    "italian": "it",
    "dutch": "nl",
    "turkish": "tr",
    "arabic": "ar",
    "hindi": "hi",
    "thai": "th",
    "vietnamese": "vi",
    "malay": "ms",
}

In [4]:
def normalize_language(value: Any) -> str:
    language = str(value).strip().lower()
    return LANGUAGE_ALIASES.get(language, language)


def download_file(url: str, destination: Path) -> Path:
    if destination.exists() and destination.stat().st_size > 0:
        return destination

    response = requests.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    destination.write_bytes(response.content)
    return destination


def load_sessions(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Parquet file not found: {path}")

    try:
        dataframe = pd.read_parquet(path, columns=PARQUET_COLUMNS)
    except Exception as exc:
        raise RuntimeError(f"Failed to read parquet file: {exc}") from exc

    missing = set(PARQUET_COLUMNS) - set(dataframe.columns)
    if missing:
        raise ValueError(f"Missing parquet columns: {sorted(missing)}")

    return dataframe


def get_session_row(dataframe: pd.DataFrame, session_id: int) -> pd.Series:
    matches = dataframe.loc[dataframe["gamesession_id"] == session_id]

    if matches.empty:
        raise ValueError(f"Session {session_id} not found")

    if len(matches) != 1:
        raise ValueError(
            f"Expected one row for session {session_id}, found {len(matches)}"
        )

    return matches.iloc[0]


FASTTEXT_MODEL = fasttext.load_model(
    str(download_file(FASTTEXT_MODEL_URL, FASTTEXT_MODEL_PATH))
)

df = load_sessions(PARQUET_PATH)
session_row = get_session_row(df, TARGET_SESSION_ID)
segments = session_row["transcript_segments"]

if segments is None:
    raise ValueError("transcript_segments is empty")

internal_lang = normalize_language(session_row["lang_detected"])
segments_to_process = list(segments) if SEGMENT_LIMIT is None else list(segments)[:SEGMENT_LIMIT]

print("Session ID          :", session_row["gamesession_id"])
print("Game                :", session_row["game_name"])
print("Internal model      :", session_row["model_type"])
print("Internal language   :", internal_lang)
print("Internal probability:", session_row["lang_probability"])
print("Total segments      :", len(segments))
print("Segments to analyze :", len(segments_to_process))

Session ID          : 140595413
Game                : Counter Strike2
Internal model      : gen10
Internal language   : ru
Internal probability: 0.9873
Total segments      : 714
Segments to analyze : 714


In [5]:
def clean_fasttext_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def fasttext_predict(
    text: str,
    k: int = FASTTEXT_TOP_K,
) -> list[dict[str, Any]]:
    text = clean_fasttext_text(text)

    if not text:
        return []

    labels_batch, probabilities_batch = FASTTEXT_MODEL.predict(
        [text],
        k=k,
    )

    labels = labels_batch[0]
    probabilities = probabilities_batch[0]

    return [
        {
            "language": normalize_language(
                label.replace("__label__", "")
            ),
            "confidence": float(probability),
        }
        for label, probability in zip(
            labels,
            probabilities,
        )
    ]


def extract_segment_tokens(segment: dict[str, Any]) -> list[dict[str, Any]]:
    raw_words = segment.get("words")

    if raw_words is None:
        text = str(segment.get("text", "")).strip()
        return [
            {"id": index, "text": token}
            for index, token in enumerate(text.split())
        ]

    tokens = []

    for index, item in enumerate(raw_words):
        if isinstance(item, dict):
            token = str(item.get("text", "")).strip()
        else:
            token = str(item).strip()

        if token:
            tokens.append({"id": index, "text": token})

    return tokens


def is_alphabetic_token(token: str) -> bool:
    return any(character.isalpha() for character in token)


def build_windows(tokens: list[dict[str, Any]]) -> list[str]:
    lexical = [item["text"] for item in tokens if is_alphabetic_token(item["text"])]

    if not lexical:
        return []

    if len(lexical) <= FASTTEXT_WINDOW_SIZE:
        return [" ".join(lexical)]

    windows = []

    for start in range(0, len(lexical), FASTTEXT_WINDOW_STRIDE):
        window = lexical[start : start + FASTTEXT_WINDOW_SIZE]

        if len(window) < max(3, FASTTEXT_WINDOW_SIZE // 2):
            break

        windows.append(" ".join(window))

    return windows


def screen_segment(segment: dict[str, Any], internal_language: str) -> dict[str, Any]:
    text = str(segment.get("text", "")).strip()
    tokens = extract_segment_tokens(segment)
    segment_predictions = fasttext_predict(text)

    primary = segment_predictions[0] if segment_predictions else {
        "language": "unknown",
        "confidence": 0.0,
    }

    window_hits = []

    for window_index, window_text in enumerate(build_windows(tokens), start=1):
        predictions = fasttext_predict(window_text)

        if not predictions:
            continue

        prediction = predictions[0]

        if (
            prediction["language"] != internal_language
            and prediction["confidence"] >= FASTTEXT_WINDOW_CONFIDENCE
        ):
            window_hits.append(
                {
                    "window": window_index,
                    "text": window_text,
                    "language": prediction["language"],
                    "confidence": prediction["confidence"],
                }
            )

    reasons = []

    if primary["language"] != internal_language:
        reasons.append("segment_language_mismatch")

    if primary["confidence"] < FASTTEXT_SEGMENT_CONFIDENCE:
        reasons.append("low_segment_confidence")

    if window_hits:
        reasons.append("foreign_window_detected")

    return {
        "text": text,
        "tokens": tokens,
        "fasttext_language": primary["language"],
        "fasttext_confidence": primary["confidence"],
        "fasttext_top_k": segment_predictions,
        "window_hits": window_hits,
        "needs_deep_audit": bool(reasons),
        "screening_reason": ", ".join(reasons),
    }


screen_rows = []
screen_details = {}

for segment_index, segment in enumerate(segments_to_process, start=1):
    segment_name = f"segment {segment_index}"
    result = screen_segment(segment, internal_lang)
    screen_details[segment_name] = result
    screen_rows.append(
        {
            "segment_index": segment_index,
            "segment": segment_name,
            "text": result["text"],
            "fasttext_language": result["fasttext_language"],
            "fasttext_confidence": result["fasttext_confidence"],
            "needs_deep_audit": result["needs_deep_audit"],
            "screening_reason": result["screening_reason"],
        }
    )

screen_df = pd.DataFrame(screen_rows)

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
    "display.max_rows", None,
):
    display(screen_df)

,segment_index,segment,text,fasttext_language,fasttext_confidence,needs_deep_audit,screening_reason
0,1,segment 1,кушать готовил очень вкусненько но приходите присаживайся реально надо подождем блин гта да ну на худа хики еще азиатик здорово привет моя киса я если что натурал сразу говорю я с азиатиком не заигрываю никогда я натурал уверенный в себе мужчина йоу я натурал уверенный,ru,0.972242,False,
1,2,segment 2,"Вот он какой маленький сидит, сладенький, вкусненький и все, Дэнс.",ru,0.804486,False,
2,3,segment 3,"Зря ты так с ним. Ну не знаю, на самом деле 50 на 50, мужики.",ru,0.960608,False,
3,4,segment 4,"50 на 50, я бы сказал бы. А у тебя что, дела? Нет, на дорожке?",ru,0.989989,False,
4,5,segment 5,"О, нифига себе. А подожди, зачем тебе дорожка? и так худая, я прям...",ru,0.900059,False,
5,6,segment 6,"Максимально я бы сказал, что ты худая. Подожди, где я должен был бы возродиться?",ru,0.998536,False,
6,7,segment 7,"Скорее всего, где-то здесь. нужно возродиться. Бля, ну если нет, нет, семейный офис.",ru,0.992197,False,
7,8,segment 8,"Нет, как будто бы начальник спал. место выхода да мне нужно вместо выхода подожди что ты пишешь хрюндель ты долбоеб хрюндель вот скажи мне честно ты дурачок или что вот если тебе ну не купили гта в детстве ты не попробовал гта это не значит что гта его ну не очень согласись не может быть что гта для тебя полная фигня я вообще никогда не поверю я знаю только одно что тебе нравится гта в целом так Дополнительное",ru,0.995237,False,
8,9,segment 9,"до свидания, я хочу поработать у вас 600 долларов на шахте!",ru,0.944353,False,
9,10,segment 10,"Здорово мужики, как у вас дела? меня GTA уже несколько лета, ну так заходи, немедленно на memphis, по правому коду плюс w у тебя будет плюс 5 тысячи долларов, плюс у тебя еще будет 7 невипки, не забывай.",ru,0.953353,False,


In [6]:
def get_openrouter_api_key() -> str:
    api_key = userdata.get("OPENROUTER_API_KEY")

    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is not available in Colab Secrets")

    return api_key


HTTP_SESSION = requests.Session()
HTTP_SESSION.headers.update(
    {
        "Authorization": f"Bearer {get_openrouter_api_key()}",
        "Content-Type": "application/json",
    }
)

TOKEN_AUDIT_SCHEMA = {
    "type": "object",
    "properties": {
        "tokens": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "kind": {
                        "type": "string",
                        "enum": ["language", "neutral", "proper_noun", "unknown"],
                    },
                    "language": {
                        "anyOf": [
                            {"type": "string", "minLength": 2, "maxLength": 8},
                            {"type": "null"},
                        ]
                    },
                },
                "required": ["id", "kind", "language"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["tokens"],
    "additionalProperties": False,
}


def build_token_audit_prompt(text: str, tokens: list[dict[str, Any]]) -> str:
    token_payload = json.dumps(tokens, ensure_ascii=False)

    return f"""
Classify each provided transcript token using its context.

Use exactly one kind per token:
- language: an ordinary lexical token that carries a language
- neutral: numbers, punctuation, symbols, standalone technical markers, or non-linguistic tokens
- proper_noun: names, usernames, brands, game titles, place names, product names, server names, or named entities
- unknown: insufficient evidence to assign a reliable category

For kind=language, return an ISO 639-1 language code when possible.
For every other kind, language must be null.

Classify usage in this utterance, not etymological origin.
Borrowed or assimilated words used as part of the surrounding language should be assigned to that surrounding language.
Do not treat a name or brand as code-switching merely because its spelling originated in another language.
Do not assign a language to numbers.
Do not create, remove, merge, split, or reorder token IDs.

Transcript:
{text}

Tokens:
{token_payload}
""".strip()


def validate_token_audit(
    response_payload: dict[str, Any],
    source_tokens: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    items = response_payload.get("tokens")

    if not isinstance(items, list):
        raise ValueError("Structured response does not contain tokens")

    expected_ids = [item["id"] for item in source_tokens]
    returned_ids = [item.get("id") for item in items]

    if returned_ids != expected_ids:
        raise ValueError("LLM token IDs do not exactly match source token IDs")

    source_by_id = {item["id"]: item["text"] for item in source_tokens}
    validated = []

    for item in items:
        token_id = item["id"]
        kind = str(item["kind"]).strip().lower()
        language = item.get("language")

        if kind == "language":
            if language is None or not str(language).strip():
                raise ValueError(f"Token {token_id} requires a language")
            language = normalize_language(language)
        else:
            language = None

        validated.append(
            {
                "id": token_id,
                "word": source_by_id[token_id],
                "kind": kind,
                "language": language,
            }
        )

    return validated


def audit_tokens_with_llm(
    text: str,
    tokens: list[dict[str, Any]],
) -> dict[str, Any]:
    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {
                "role": "user",
                "content": build_token_audit_prompt(text, tokens),
            }
        ],
        "provider": {"require_parameters": True},
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": "token_language_audit",
                "strict": True,
                "schema": TOKEN_AUDIT_SCHEMA,
            },
        },
        "plugins": [{"id": "response-healing"}],
    }

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        started_at = time.perf_counter()

        try:
            response = HTTP_SESSION.post(
                OPENROUTER_API_URL,
                json=payload,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            response.raise_for_status()
            response_json = response.json()
            content = response_json["choices"][0]["message"]["content"]
            parsed = json.loads(content)
            validated = validate_token_audit(parsed, tokens)

            return {
                "tokens": validated,
                "response_id": response_json.get("id"),
                "model": response_json.get("model", OPENROUTER_MODEL),
                "usage": response_json.get("usage", {}),
                "elapsed_seconds": time.perf_counter() - started_at,
            }

        except (
            requests.RequestException,
            KeyError,
            IndexError,
            TypeError,
            ValueError,
            json.JSONDecodeError,
        ) as exc:
            last_error = exc

            if attempt == MAX_RETRIES:
                break

            delay = RETRY_BACKOFF_SECONDS * (2 ** (attempt - 1))
            delay += random.uniform(0, 0.5)
            time.sleep(delay)

    raise RuntimeError(
        f"LLM request failed after {MAX_RETRIES} attempts: {last_error}"
    )

In [7]:
def summarize_audited_tokens(
    audited_tokens: list[dict[str, Any]],
) -> dict[str, Any]:
    language_tokens = [
        item for item in audited_tokens if item["kind"] == "language"
    ]

    excluded_tokens = [
        item for item in audited_tokens if item["kind"] != "language"
    ]

    counts = Counter(item["language"] for item in language_tokens)
    total_language_tokens = sum(counts.values())
    words_by_language = defaultdict(list)

    for item in language_tokens:
        words_by_language[item["language"]].append(item["word"])

    percentages = {
        language: count / total_language_tokens * 100
        for language, count in counts.items()
    } if total_language_tokens else {}

    excluded_by_kind = defaultdict(list)

    for item in excluded_tokens:
        excluded_by_kind[item["kind"]].append(item["word"])

    return {
        "counts": dict(counts),
        "percentages": percentages,
        "words_by_language": dict(words_by_language),
        "excluded_by_kind": dict(excluded_by_kind),
        "language_token_count": total_language_tokens,
    }


def lexical_token_count(tokens: list[dict[str, Any]]) -> int:
    return sum(is_alphabetic_token(item["text"]) for item in tokens)


def run_hybrid_audit(
    screen_dataframe: pd.DataFrame,
    screen_detail_map: dict[str, Any],
    internal_language: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    rows = []
    token_rows = []
    request_rows = []
    deep_details = {}

    total_segments = len(screen_dataframe)

    for position, row in enumerate(screen_dataframe.itertuples(index=False), start=1):
        detail = screen_detail_map[row.segment]

        if not row.needs_deep_audit:
            word_count = lexical_token_count(detail["tokens"])
            rows.append(
                {
                    "segment_index": row.segment_index,
                    "segment": row.segment,
                    "text": row.text,
                    "source": "fasttext_clear",
                    "language": row.fasttext_language,
                    "percentage": 100.0,
                    "word_count": word_count,
                    "mismatch_words": "",
                    "excluded_words": "",
                    "fasttext_language": row.fasttext_language,
                    "fasttext_confidence": row.fasttext_confidence,
                    "screening_reason": row.screening_reason,
                }
            )
            continue

        print(f"[{position}/{total_segments}] Deep audit {row.segment}")

        audit = audit_tokens_with_llm(row.text, detail["tokens"])
        summary = summarize_audited_tokens(audit["tokens"])
        deep_details[row.segment] = {
            "audit": audit,
            "summary": summary,
        }

        request_rows.append(
            {
                "segment": row.segment,
                "response_id": audit["response_id"],
                "model": audit["model"],
                "elapsed_seconds": audit["elapsed_seconds"],
                "prompt_tokens": audit["usage"].get("prompt_tokens"),
                "completion_tokens": audit["usage"].get("completion_tokens"),
                "total_tokens": audit["usage"].get("total_tokens"),
            }
        )

        for item in audit["tokens"]:
            token_rows.append(
                {
                    "segment_index": row.segment_index,
                    "segment": row.segment,
                    "text": row.text,
                    **item,
                }
            )

        excluded_words = "; ".join(
            f"{kind}: {', '.join(words)}"
            for kind, words in summary["excluded_by_kind"].items()
            if words
        )

        if not summary["percentages"]:
            rows.append(
                {
                    "segment_index": row.segment_index,
                    "segment": row.segment,
                    "text": row.text,
                    "source": "llm_audit",
                    "language": "unknown",
                    "percentage": 100.0,
                    "word_count": 0,
                    "mismatch_words": "",
                    "excluded_words": excluded_words,
                    "fasttext_language": row.fasttext_language,
                    "fasttext_confidence": row.fasttext_confidence,
                    "screening_reason": row.screening_reason,
                }
            )
            continue

        for language, percentage in summary["percentages"].items():
            words = summary["words_by_language"].get(language, [])
            rows.append(
                {
                    "segment_index": row.segment_index,
                    "segment": row.segment,
                    "text": row.text,
                    "source": "llm_audit",
                    "language": language,
                    "percentage": percentage,
                    "word_count": summary["counts"][language],
                    "mismatch_words": (
                        "" if language == internal_language else ", ".join(words)
                    ),
                    "excluded_words": excluded_words,
                    "fasttext_language": row.fasttext_language,
                    "fasttext_confidence": row.fasttext_confidence,
                    "screening_reason": row.screening_reason,
                }
            )

    result_df = pd.DataFrame(rows)
    token_df = pd.DataFrame(token_rows)
    request_df = pd.DataFrame(request_rows)

    if not result_df.empty:
        result_df["is_internal_language"] = (
            result_df["language"] == internal_language
        )
        result_df["material_mismatch"] = (
            ~result_df["is_internal_language"]
            & (result_df["language"] != "unknown")
            & (result_df["word_count"] >= MIN_MISMATCH_WORDS)
            & (result_df["percentage"] >= MIN_MISMATCH_PERCENTAGE)
        )
        result_df = result_df.sort_values(
            by=[
                "material_mismatch",
                "is_internal_language",
                "percentage",
                "language",
                "segment_index",
            ],
            ascending=[False, True, False, True, True],
        ).reset_index(drop=True)

    return result_df, token_df, request_df, deep_details


result_df, token_df, request_df, deep_details = run_hybrid_audit(
    screen_df,
    screen_details,
    internal_lang,
)

[18/714] Deep audit segment 18
[29/714] Deep audit segment 29
[96/714] Deep audit segment 96
[111/714] Deep audit segment 111
[137/714] Deep audit segment 137
[186/714] Deep audit segment 186
[211/714] Deep audit segment 211
[232/714] Deep audit segment 232
[238/714] Deep audit segment 238
[289/714] Deep audit segment 289
[296/714] Deep audit segment 296
[307/714] Deep audit segment 307
[309/714] Deep audit segment 309
[310/714] Deep audit segment 310
[350/714] Deep audit segment 350
[399/714] Deep audit segment 399
[427/714] Deep audit segment 427
[452/714] Deep audit segment 452
[468/714] Deep audit segment 468
[488/714] Deep audit segment 488
[520/714] Deep audit segment 520
[573/714] Deep audit segment 573
[593/714] Deep audit segment 593
[600/714] Deep audit segment 600
[689/714] Deep audit segment 689
[708/714] Deep audit segment 708


In [8]:
display_df = result_df.copy()

if not display_df.empty:
    display_df["percentage"] = display_df["percentage"].map(
        lambda value: f"{value:.1f}%"
    )
    display_df["fasttext_confidence"] = display_df["fasttext_confidence"].map(
        lambda value: f"{value:.3f}"
    )
    display_df = display_df[
        [
            "segment",
            "text",
            "source",
            "language",
            "percentage",
            "word_count",
            "material_mismatch",
            "mismatch_words",
            "excluded_words",
            "fasttext_language",
            "fasttext_confidence",
            "screening_reason",
        ]
    ]

print("Session ID       :", session_row["gamesession_id"])
print("Internal language:", internal_lang)

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
    "display.max_rows", None,
):
    display(display_df)

Session ID       : 140595413
Internal language: ru


,segment,text,source,language,percentage,word_count,material_mismatch,mismatch_words,excluded_words,fasttext_language,fasttext_confidence,screening_reason
0,segment 1,кушать готовил очень вкусненько но приходите присаживайся реально надо подождем блин гта да ну на худа хики еще азиатик здорово привет моя киса я если что натурал сразу говорю я с азиатиком не заигрываю никогда я натурал уверенный в себе мужчина йоу я натурал уверенный,fasttext_clear,ru,100.0%,45,False,,,ru,0.972,
1,segment 2,"Вот он какой маленький сидит, сладенький, вкусненький и все, Дэнс.",fasttext_clear,ru,100.0%,10,False,,,ru,0.804,
2,segment 3,"Зря ты так с ним. Ну не знаю, на самом деле 50 на 50, мужики.",fasttext_clear,ru,100.0%,13,False,,,ru,0.961,
3,segment 4,"50 на 50, я бы сказал бы. А у тебя что, дела? Нет, на дорожке?",fasttext_clear,ru,100.0%,13,False,,,ru,0.990,
4,segment 5,"О, нифига себе. А подожди, зачем тебе дорожка? и так худая, я прям...",fasttext_clear,ru,100.0%,13,False,,,ru,0.900,
5,segment 6,"Максимально я бы сказал, что ты худая. Подожди, где я должен был бы возродиться?",fasttext_clear,ru,100.0%,14,False,,,ru,0.999,
6,segment 7,"Скорее всего, где-то здесь. нужно возродиться. Бля, ну если нет, нет, семейный офис.",fasttext_clear,ru,100.0%,14,False,,,ru,0.992,
7,segment 8,"Нет, как будто бы начальник спал. место выхода да мне нужно вместо выхода подожди что ты пишешь хрюндель ты долбоеб хрюндель вот скажи мне честно ты дурачок или что вот если тебе ну не купили гта в детстве ты не попробовал гта это не значит что гта его ну не очень согласись не может быть что гта для тебя полная фигня я вообще никогда не поверю я знаю только одно что тебе нравится гта в целом так Дополнительное",fasttext_clear,ru,100.0%,79,False,,,ru,0.995,
8,segment 9,"до свидания, я хочу поработать у вас 600 долларов на шахте!",fasttext_clear,ru,100.0%,11,False,,,ru,0.944,
9,segment 10,"Здорово мужики, как у вас дела? меня GTA уже несколько лета, ну так заходи, немедленно на memphis, по правому коду плюс w у тебя будет плюс 5 тысячи долларов, плюс у тебя еще будет 7 невипки, не забывай.",fasttext_clear,ru,100.0%,36,False,,,ru,0.953,


In [9]:
mismatch_df = result_df.loc[
    ~result_df["is_internal_language"]
    & (result_df["language"] != "unknown")
].copy()

if not mismatch_df.empty:
    mismatch_df["percentage"] = mismatch_df["percentage"].map(
        lambda value: f"{value:.1f}%"
    )
    mismatch_df["fasttext_confidence"] = mismatch_df["fasttext_confidence"].map(
        lambda value: f"{value:.3f}"
    )
    mismatch_df = mismatch_df[
        [
            "segment",
            "text",
            "language",
            "percentage",
            "word_count",
            "material_mismatch",
            "mismatch_words",
            "excluded_words",
            "fasttext_language",
            "fasttext_confidence",
            "screening_reason",
        ]
    ]

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
    "display.max_rows", None,
):
    display(mismatch_df)

,segment_index,segment,text,source,language,percentage,word_count,mismatch_words,excluded_words,fasttext_language,fasttext_confidence,screening_reason,is_internal_language,material_mismatch


In [10]:
def build_session_summary(result_dataframe: pd.DataFrame) -> pd.DataFrame:
    valid = result_dataframe.loc[
        result_dataframe["language"] != "unknown"
    ].copy()

    if valid.empty:
        return pd.DataFrame(
            columns=["language", "word_count", "percentage"]
        )

    summary = (
        valid.groupby("language", as_index=False)["word_count"]
        .sum()
        .sort_values("word_count", ascending=False)
        .reset_index(drop=True)
    )

    total = summary["word_count"].sum()
    summary["percentage"] = summary["word_count"] / total * 100 if total else 0.0
    return summary


session_summary_df = build_session_summary(result_df)
session_summary_display_df = session_summary_df.copy()

if not session_summary_display_df.empty:
    session_summary_display_df["percentage"] = session_summary_display_df[
        "percentage"
    ].map(lambda value: f"{value:.1f}%")

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
    "display.max_rows", None,
):
    display(session_summary_display_df)

,language,word_count,percentage
0,ru,14666,100.0%


In [11]:
if not token_df.empty:
    with pd.option_context(
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.width", None,
        "display.max_rows", None,
    ):
        display(token_df)

if not request_df.empty:
    with pd.option_context(
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.width", None,
        "display.max_rows", None,
    ):
        display(request_df)

,segment_index,segment,text,id,word,kind,language
0,18,segment 18,"А кто-то говорил, я на матче, да не, да не, я такого не говорил.",0,А,language,ru
1,18,segment 18,"А кто-то говорил, я на матче, да не, да не, я такого не говорил.",1,кто,language,ru
2,18,segment 18,"А кто-то говорил, я на матче, да не, да не, я такого не говорил.",2,-то,language,ru
3,18,segment 18,"А кто-то говорил, я на матче, да не, да не, я такого не говорил.",3,"говорил,",language,ru
4,18,segment 18,"А кто-то говорил, я на матче, да не, да не, я такого не говорил.",4,я,language,ru
5,18,segment 18,"А кто-то говорил, я на матче, да не, да не, я такого не говорил.",5,на,language,ru
6,18,segment 18,"А кто-то говорил, я на матче, да не, да не, я такого не говорил.",6,"матче,",language,ru
7,18,segment 18,"А кто-то говорил, я на матче, да не, да не, я такого не говорил.",7,да,language,ru
8,18,segment 18,"А кто-то говорил, я на матче, да не, да не, я такого не говорил.",8,"не,",language,ru
9,18,segment 18,"А кто-то говорил, я на матче, да не, да не, я такого не говорил.",9,да,language,ru


,segment,response_id,model,elapsed_seconds,prompt_tokens,completion_tokens,total_tokens
0,segment 18,gen-1787408738-iyRCRKCrXZlFb5oKJjUz,openai/gpt-5.4,2.503039,489,195,684
1,segment 29,gen-1787408740-dEZpyVq8McHt7H81uHT6,openai/gpt-5.4,2.164592,500,195,695
2,segment 96,gen-1787408742-XAtYp8AnCXGVrWNqlF3H,openai/gpt-5.4,1.449536,341,63,404
3,segment 111,gen-1787408744-obWtpQoOwZXhzESqL8rN,openai/gpt-5.4,2.105518,416,123,539
4,segment 137,gen-1787408746-WlWPQK0lw86uc33CS4iB,openai/gpt-5.4,1.643632,378,87,465
5,segment 186,gen-1787408747-olwgtztURR1DEA6sae9x,openai/gpt-5.4,4.323382,1046,631,1677
6,segment 211,gen-1787408752-Svgyteg6lKp08Nyq5R2i,openai/gpt-5.4,1.427389,373,87,460
7,segment 232,gen-1787408753-W0BB3apeEalgdMaLX4wo,openai/gpt-5.4,3.062924,495,171,666
8,segment 238,gen-1787408756-hvuqyQ181ohU7TzQh8yi,openai/gpt-5.4,1.570371,393,99,492
9,segment 289,gen-1787408758-uemgrVMY8MQF12RrEVXs,openai/gpt-5.4,1.947649,333,51,384


In [12]:
OUTPUT_DIR = Path("/content/language_audit_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

result_df.to_parquet(
    OUTPUT_DIR / f"session_{TARGET_SESSION_ID}_language_audit.parquet",
    index=False,
)

screen_df.to_parquet(
    OUTPUT_DIR / f"session_{TARGET_SESSION_ID}_screening.parquet",
    index=False,
)

if not token_df.empty:
    token_df.to_parquet(
        OUTPUT_DIR / f"session_{TARGET_SESSION_ID}_token_audit.parquet",
        index=False,
    )

if not request_df.empty:
    request_df.to_parquet(
        OUTPUT_DIR / f"session_{TARGET_SESSION_ID}_request_log.parquet",
        index=False,
    )

session_summary_df.to_parquet(
    OUTPUT_DIR / f"session_{TARGET_SESSION_ID}_summary.parquet",
    index=False,
)

print(OUTPUT_DIR)

/content/language_audit_output
